<a href="https://colab.research.google.com/github/NikoriakViktot/PY-Course-Victor-Nikoriak-22-09-2026/blob/main/module_1/lessons/lesson_07_functions/note_lesson_07_functions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Урок 7 — Функції

> За цей урок: замість того щоб копіювати той самий код по кілька разів (Урок 6 — Minesweeper одним суцільним блоком), навчаєшся розбивати програму на іменовані, багаторазові шматки — `def`, параметри, `return`, декомпозиція. Головна вправа уроку — **рефакторинг** реальної гри «Minesweeper» з плаского скрипту у функції, крок за кроком, з перевіркою на кожному кроці.

Структура уроку: **RETRIEVE → CONCEPT → PREDICT/RUN/INVESTIGATE/MODIFY → CREATE → TRANSFER** (та сама послідовність, що й в Уроці 4).

## 🔁 RETRIEVE — пригадай Урок 6 (без підглядання)

Урок 6 був про цикли, словники та comprehensions. Дай відповідь усно чи на папері, **не запускаючи нічого**:

1. Що поверне `for k, v in {'a': 1}:`  — спрацює нормально чи впаде з помилкою? Якщо впаде — з якою?
2. Як одним рядком (comprehension) отримати список квадратів **парних** чисел від 1 до 10?
3. `d.setdefault('x', []).append(5)` — що робить цей рядок, якщо ключа `'x'` в `d` ще немає?

Звір відповіді нижче.

<details>
<summary>Відповіді</summary>

1. <code>ValueError</code> — <code>for k, v in d</code> (без <code>.items()</code>) ітерує по <b>ключах</b>, а ключ <code>'a'</code> — рядок з одного символу, який не можна розпакувати у дві змінні <code>k, v</code>.
2. <code>[x**2 for x in range(1, 11) if x % 2 == 0]</code>
3. Створює <code>d['x'] = []</code>, а потім одразу додає до нього <code>5</code> — після рядка <code>d == {'x': [5]}</code>.

</details>

## 📖 CONCEPT

### 1. Навіщо функції?

Той самий код, повторений кілька разів — проблема: якщо треба виправити логіку, доведеться шукати й правити **усі** копії.

In [ ]:
# БЕЗ функції — та сама формула переказана тричі
balance1 = 1000
interest1 = balance1 * 0.05
balance1_after = balance1 + interest1

balance2 = 2500
interest2 = balance2 * 0.05   # та сама формула — дублювання
balance2_after = balance2 + interest2

print(f"БЕЗ функції: {balance1_after}, {balance2_after}")

# З функцією — логіка в одному місці
def apply_interest(balance, rate=0.05):
    """Повертає баланс після нарахування відсотків."""
    return balance + balance * rate

print("З функцією:", apply_interest(1000), apply_interest(2500))
print("Інша ставка:", apply_interest(1000, rate=0.10))

### 2. Синтаксис: визначення, параметри, `return` vs `print`

```
def  назва(param1, param2=значення_за_замовчуванням):   ← сигнатура
    тіло функції
    return результат                                     ← ОБОВ'ЯЗКОВО return, якщо
                                                             результат потрібен ДАЛІ в коді
```

- **Параметр** — ім'я в дужках при *визначенні* (`def f(x):`). **Аргумент** — реальне значення при *виклику* (`f(5)`).
- Параметр зі значенням за замовчуванням (`rate=0.05`) можна не вказувати при виклику.
- `print()` показує значення **людині** на екрані. `return` передає значення **назад у програму**, щоб зберегти в змінну й використати далі. Функція без `return` завжди повертає `None`.

In [ ]:
# print замість return — типова пастка
def bad_double(x):
    print(x * 2)     # тільки показує

def good_double(x):
    return x * 2      # повертає назад у програму

result_bad = bad_double(5)     # надрукує 10, але result_bad = None
result_good = good_double(5)   # нічого не друкує, result_good = 10

print("result_bad  =", result_bad)
print("result_good =", result_good)
print("result_good * 3 =", result_good * 3)   # можна використати далі

try:
    result_bad * 3
except TypeError as e:
    print("result_bad * 3 -> TypeError:", e)

### 3. Декомпозиція

**Принцип єдиної відповідальності:** одна функція — одна задача. Якщо функцію важко назвати одним дієсловом — вона, ймовірно, робить забагато.

Маленький бонус: функція, що не змінює нічого поза собою (не мутує вхідні дані, не читає/пише глобальні змінні) і при однакових аргументах завжди повертає однаковий результат, називається **чистою** — її найлегше тестувати і найбезпечніше повторно використовувати.

In [ ]:
# Декомпозиція: велика задача -> маленькі одноцільові функції
def is_low_stock(quantity, threshold=5):
    """Предикат: чи товару залишилось мало."""
    return quantity <= threshold


def restock_amount(quantity, target=20):
    """Трансформер: скільки треба замовити, щоб дійти до target."""
    return max(0, target - quantity)


def inventory_report(stock):
    """Декомпозиція: використовує обидві функції вище для повного звіту."""
    low = {item: qty for item, qty in stock.items() if is_low_stock(qty)}
    orders = {item: restock_amount(qty) for item, qty in low.items()}
    return {"low_stock": low, "to_order": orders}


stock = {"олівці": 3, "зошити": 40, "лінійки": 5, "ручки": 12}
report = inventory_report(stock)
print("Мало на складі:", report["low_stock"])
print("Замовити:", report["to_order"])

### 4. Три патерни: Predicate / Transformer / Reducer

| Патерн | Питання | Вхід → Вихід | Приклад |
|---|---|---|---|
| **Predicate** | Так/Ні? | 1 елемент → `bool` | `is_low_stock(qty)` |
| **Transformer** | Як змінити форму? | 1 елемент → 1 елемент (нової форми) | `restock_amount(qty)` |
| **Reducer** | Як зібрати в одне? | багато елементів → 1 значення | `sum(...)`, `max(...)` |

Ці три питання — універсальний спосіб підійти до будь-якої задачі обробки списку даних, і саме вони знадобляться нижче в CREATE і TRANSFER.

In [ ]:
# Три патерни разом на одному наборі даних
scores = [55, 82, 78, 92, 45, 88, 63]

def is_passed(score):          # PREDICATE
    return score >= 60

def to_letter(score):          # TRANSFORMER
    if score >= 90: return "A"
    if score >= 80: return "B"
    if score >= 70: return "C"
    return "D"

passed = [s for s in scores if is_passed(s)]         # PREDICATE у comprehension
letters = [to_letter(s) for s in passed]              # TRANSFORMER у comprehension
average = sum(passed) / len(passed)                    # REDUCER

print("Здали:", passed)
print("Оцінки:", letters)
print(f"Середня серед здали: {average:.1f}")

### Що свідомо НЕ увійшло в цей конспект

Формат уроку — приблизно 2 години, і більшість цього часу піде на керовану вправу нижче (рефакторинг Minesweeper). Тому з повного конспекту `lesson_07_functions/notes_functions.ipynb` свідомо **вирізано**:

- детальний розбір **stack frame** (окремий блок з покроковою діаграмою пам'яті) — залишено лише одна згадка про ізоляцію локальних змінних, без окремої демонстрації;
- **`*args`** (змінна кількість аргументів) — жодна функція в Minesweeper-рефакторингу його не потребує;
- **чисті функції** як окрема велика тема (impure vs pure, `.sort()` vs `sorted()`) — залишено лише одне речення в п.3;
- повний **pipeline** (Filter → Map → Reduce з генераторами та порівнянням пам'яті) — залишено лише сама ідея трьох патернів;
- каталог **типових помилок** (global-змінні, мутація тощо) як окремий розділ;
- підсумкова **шпаргалка**.

Хто хоче побачити ці теми повністю — вони є в `lesson_07_functions/notes_functions.ipynb` (старий конспект, глибший за обсягом, ніж потрібно для цього формату уроку).

## 🎮 PREDICT / RUN / INVESTIGATE / MODIFY — рефакторинг Minesweeper

Головна вправа уроку. На Уроці 6 ви вже бачили гру «Minesweeper», написану одним суцільним скриптом — файл [`resources/minesweeper_before_refactor.py`](https://github.com/NikoriakViktot/PY-Course-Victor-Nikoriak-22-09-2026/blob/main/module_1/lessons/lesson_07_functions/resources/minesweeper_before_refactor.py) поруч із цим конспектом. Задача цього блоку — розкласти її на функції **крок за кроком**, і на кожному кроці **перевірити**, що поведінка не зламалась — а не просто повірити на слово.

Спочатку — інструменти, якими ми будемо симулювати гру без реальної клавіатури (той самий прийом, що на Уроці 4 для PIN-коду, лише узагальнений).

In [ ]:
import io
import os
import random
import contextlib

SIZE = 8
BOMBS = 10


@contextlib.contextmanager
def canned_inputs(responses):
    """Тимчасово підміняє вбудований input() списком заготовлених відповідей."""
    import builtins
    it = iter(responses)
    original_input = builtins.input

    def fake_input(prompt=""):
        try:
            return next(it)
        except StopIteration:
            raise EOFError("Скінчились заготовлені відповіді")

    builtins.input = fake_input
    try:
        yield
    finally:
        builtins.input = original_input


print("Готово: canned_inputs(...) — контекст-менеджер для симуляції input()")

### PREDICT

Відкрий `resources/minesweeper_before_refactor.py` (клітинка нижче також друкує його вміст). Перш ніж запускати щось — дай відповідь:

1. Змінні `row`/`col` зустрічаються у **двох різних** місцях файлу (у циклі друку дошки і в циклі читання ходу). Чому це взагалі спрацьовує без конфлікту, і чим це небезпечно при подальшому рефакторингу?
2. Що станеться, якщо гравець введе координати клітинки, яку вже відкрито раніше?

<details>
<summary>Відповідь</summary>

1. Спрацьовує, бо весь код лежить на **одному рівні модуля** — це один спільний простір імен, і останній `row`/`col`, які були присвоєні, "перекривають" попередні. Небезпечно це стає, коли код зростає: два незалежні шматки логіки випадково діляться іменами змінних, і зміна одного може непомітно зламати інший. Коли ми розкладемо код у функції — кожна функція отримає свій **власний** простір імен (свій stack frame), і ця проблема зникне сама собою.
2. Спрацьовує перевірка `if hidden[row][col] != ".":` → друкується `"This cell is already open"` і `continue` — гравцю показують дошку знову, той самий хід не рахується.

</details>

In [ ]:
with open("resources/minesweeper_before_refactor.py", encoding="utf-8") as f:
    before_source = f.read()

print(before_source)

### RUN

Запускаємо `minesweeper_before_refactor.py` "наосліп" — з фіксованим `random.seed()` і заготовленим списком ходів замість реальної клавіатури. Послідовність ходів навмисно містить: повторний хід (дублікат), хід поза межами дошки і некоректний ввід — щоб перевірити всі три `if`/`elif`/`else` гілки одразу.

In [ ]:
seed = 42
canned_moves = ["0 0", "0 0", "9 9", "abc x", "1 1", "2 2", "3 3",
                "0 1", "1 0", "7 7", "6 6", "5 5"]

buffer_before = io.StringIO()
random.seed(seed)
with canned_inputs(canned_moves), contextlib.redirect_stdout(buffer_before):
    exec(compile(before_source, "minesweeper_before_refactor.py", "exec"), {})

output_before = buffer_before.getvalue()
print(output_before)

### INVESTIGATE

У виведеному коді видно кілька природних блоків, кожен із чітким входом і виходом — саме такі блоки й стають функціями:

| Блок логіки | Майбутня функція | Вхід | Вихід |
|---|---|---|---|
| Генерація позицій бомб | `create_bombs(size, count=10)` | `size`, `count` | множина `(row, col)` |
| Підрахунок бомб навколо клітинки | `count_around(bombs, row, col)` | бомби, координати | число |
| Створення порожньої дошки | `create_board(size)` | `size` | список списків |
| Друк дошки | `print_board(board)` | дошка | (нічого — друкує) |
| Зчитування ходу гравця | `read_move(size)` | `size` | `(row, col)` |
| Показ усіх бомб наприкінці | `show_bombs(board, bombs)` | дошка, бомби | (нічого — мутує дошку) |

Це саме та декомпозиція, яку використовує еталонний розв'язок викладача. Нижче — п'ять контрольних точок: кожна визначає одну функцію і одразу її перевіряє, перш ніж рухатись далі.

### MODIFY — Контрольна точка 1: `create_bombs` (розібраний приклад)

Перша функція вже готова — прочитай і запусти, нічого міняти не треба.

In [ ]:
def create_bombs(size, count=10):
    """Повертає множину (row, col) — випадкові унікальні позиції бомб."""
    bombs = set()
    while len(bombs) < count:
        bombs.add((random.randint(0, size - 1), random.randint(0, size - 1)))
    return bombs


random.seed(1)
test_bombs_cp1 = create_bombs(8, 10)
print(test_bombs_cp1)

assert len(test_bombs_cp1) == 10
assert all(0 <= r < 8 and 0 <= c < 8 for r, c in test_bombs_cp1)
print("OK — 10 унікальних бомб у межах дошки 8x8")

### Контрольна точка 2: `count_around` (заповни пропуск)

Порахувати, скільки бомб торкається клітинки `(row, col)` — перевіряємо всі 9 клітинок квадрата 3×3 навколо неї (включно з самою клітинкою — так само, як в еталонному розв'язку).

In [ ]:
def count_around(bombs, row, col):
    """Скільки бомб торкається клітинки (row, col), включно з нею самою."""
    around = 0
    # TODO: пройдись подвійним циклом dr, dc по (-1, 0, 1) x (-1, 0, 1)
    # і для кожної пари перевір, чи (row + dr, col + dc) є в bombs
    # BEGIN SOLUTION
    for dr in (-1, 0, 1):
        for dc in (-1, 0, 1):
            if (row + dr, col + dc) in bombs:
                around += 1
    # END SOLUTION
    return around


test_bombs_cp2 = {(1, 1), (1, 2), (3, 3)}
print("count_around(1,1) =", count_around(test_bombs_cp2, 1, 1))
print("count_around(0,0) =", count_around(test_bombs_cp2, 0, 0))
print("count_around(5,5) =", count_around(test_bombs_cp2, 5, 5))
print("count_around(2,2) =", count_around(test_bombs_cp2, 2, 2))

assert count_around(test_bombs_cp2, 1, 1) == 2   # сама клітинка (1,1) теж бомба + сусідня (1,2)
assert count_around(test_bombs_cp2, 0, 0) == 1
assert count_around(test_bombs_cp2, 5, 5) == 0
assert count_around(test_bombs_cp2, 2, 2) == 3
print("OK")

### Контрольна точка 3: `create_board` + `print_board` (заповни пропуски)

In [ ]:
def create_board(size):
    """Повертає нову приховану дошку — список списків із крапками '.'."""
    # BEGIN SOLUTION
    return [["." for col in range(size)] for row in range(size)]
    # END SOLUTION


def print_board(board):
    """Друкує дошку з номерами рядків і колонок."""
    header = "  "
    for col in range(len(board)):
        header += " " + str(col)
    print(header)
    # BEGIN SOLUTION
    for row in range(len(board)):
        line = str(row) + " "
        for cell in board[row]:
            line += " " + cell
        print(line)
    # END SOLUTION


test_board_cp3 = create_board(3)
buf_cp3 = io.StringIO()
with contextlib.redirect_stdout(buf_cp3):
    print_board(test_board_cp3)
printed_cp3 = buf_cp3.getvalue()
print(printed_cp3)

expected_cp3 = "   0 1 2\n0  . . .\n1  . . .\n2  . . .\n"
assert printed_cp3 == expected_cp3
print("OK")

### Контрольна точка 4: `read_move` (розібраний приклад)

Це переважно "проводка" (введення + перевірка формату), а не нова ідея — тому дано готовою. Перевіряємо, що вона коректно пропускає невалідний ввід і хід поза межами дошки, перш ніж прийняти правильний.

In [ ]:
def read_move(size):
    """Питає хід, поки не введуть коректні координати. Повертає (row, col)."""
    while True:
        answer = input("Row and column, for example 3 5: ").split()
        if len(answer) != 2 or not answer[0].isdigit() or not answer[1].isdigit():
            print("Type two numbers from 0 to", size - 1)
        elif int(answer[0]) >= size or int(answer[1]) >= size:
            print("This cell is outside the board")
        else:
            return int(answer[0]), int(answer[1])


with canned_inputs(["abc x", "9 9", "1 1"]):
    move_cp4 = read_move(8)

print("move =", move_cp4)
assert move_cp4 == (1, 1)
print("OK — некоректний формат і вихід за межі дошки коректно пропущені")

### Контрольна точка 5: `show_bombs` (заповни пропуск)

In [ ]:
def show_bombs(board, bombs):
    """Позначає '*' на кожній клітинці, де є бомба (мутує board на місці)."""
    # BEGIN SOLUTION
    for row, col in bombs:
        board[row][col] = "*"
    # END SOLUTION


test_board_cp5 = create_board(3)
show_bombs(test_board_cp5, {(0, 0), (2, 2)})
print(test_board_cp5)

assert test_board_cp5[0][0] == "*"
assert test_board_cp5[2][2] == "*"
assert test_board_cp5[0][1] == "."
print("OK")

### Збірка: повна гра з функцій

Усі п'ять функцій готові — тепер складаємо з них той самий ігровий цикл, що був у `minesweeper_before_refactor.py`, але вже через виклики функцій замість плаского коду.

In [ ]:
def play_minesweeper_after(size=SIZE, bomb_count=BOMBS):
    bombs = create_bombs(size, bomb_count)
    hidden = create_board(size)
    opened = 0

    while opened < size * size - bomb_count:
        print_board(hidden)
        row, col = read_move(size)
        if hidden[row][col] != ".":
            print("This cell is already open")
            continue
        if (row, col) in bombs:
            print("Boom! Game over.")
            break
        hidden[row][col] = str(count_around(bombs, row, col))
        opened += 1
    else:
        print("You win!")

    show_bombs(hidden, bombs)
    print_board(hidden)


print("play_minesweeper_after визначено")

### Найризикованіша перевірка: «до» і «після» дають ІДЕНТИЧНИЙ вивід

Запускаємо `play_minesweeper_after()` з тим самим `seed` і тими самими ходами, що й "до"-версію вище, і порівнюємо вивід **побайтово**.

In [ ]:
buffer_after = io.StringIO()
random.seed(seed)
with canned_inputs(canned_moves), contextlib.redirect_stdout(buffer_after):
    play_minesweeper_after()

output_after = buffer_after.getvalue()
print(output_after)

assert output_after == output_before, "Рефакторинг змінив поведінку гри!"
print("\n✅ Вивід «до» і «після» повністю ідентичний — рефакторинг зберіг поведінку гри.")

## 🛠️ CREATE — самостійна декомпозиція

Дано плаский скрипт обрахунку бібліотечних штрафів. Розклади його на функції за тими самими трьома патернами:

- `is_overdue(days_late)` — **predicate**: чи є прострочення
- `calculate_fine(days_late, rate=FINE_PER_DAY)` — **transformer**: скільки грн штрафу за це позичення
- `total_fine(loans, rate=FINE_PER_DAY)` — **reducer**: загальний штраф по всіх позиченнях

In [ ]:
loans = [
    ("Кобзар", 0),
    ("1984", 5),
    ("Тіні забутих предків", 12),
    ("Захар Беркут", 0),
    ("Момент істини", 20),
]

FINE_PER_DAY = 2  # грн за день прострочення

# YOUR CODE HERE
# BEGIN SOLUTION
def is_overdue(days_late):
    return days_late > 0


def calculate_fine(days_late, rate=FINE_PER_DAY):
    return days_late * rate


def total_fine(loans, rate=FINE_PER_DAY):
    return sum(calculate_fine(days, rate) for _, days in loans if is_overdue(days))
# END SOLUTION

for title, days_late in loans:
    if is_overdue(days_late):
        print(f"{title}: прострочено на {days_late} дн., штраф {calculate_fine(days_late)} грн")
    else:
        print(f"{title}: без прострочення")

print(f"Загальний штраф: {total_fine(loans)} грн")

assert calculate_fine(5) == 10
assert calculate_fine(20) == 40
assert total_fine(loans) == 74
print("OK")

## 🔄 TRANSFER — та сама структура, інші дані

Той самий набір із трьох питань (predicate / transformer / reducer), але тепер про список треків плейлиста — **інша поверхнева деталь**, та сама структура рішення:

- `is_popular(plays)` — **predicate**: чи прослуховувань ≥ `MIN_POPULAR_PLAYS`
- `format_duration(seconds)` — **transformer**: секунди → рядок `"хв:сс"`
- підсумкова тривалість **популярних** треків — **reducer**

In [ ]:
tracks = [
    ("Ой у лузі червона калина", 187, 152000),
    ("Тримай", 203, 89000),
    ("Незалежність", 245, 310000),
    ("Спокій", 165, 12000),
    ("Гуцулка Ксеня", 198, 45000),
]  # (назва, тривалість_сек, кількість_прослуховувань)

MIN_POPULAR_PLAYS = 50000

# YOUR CODE HERE
# BEGIN SOLUTION
def is_popular(plays):
    return plays >= MIN_POPULAR_PLAYS


def format_duration(seconds):
    minutes, secs = divmod(seconds, 60)
    return f"{minutes}:{secs:02d}"


popular_tracks = [t for t in tracks if is_popular(t[2])]
total_popular_duration = sum(t[1] for t in popular_tracks)
# END SOLUTION

for title, duration, plays in tracks:
    marker = "★" if is_popular(plays) else " "
    print(f"{marker} {title:<30} {format_duration(duration)}  ({plays} прослуховувань)")

print()
print(f"Популярних треків: {len(popular_tracks)}")
print(f"Їхня сумарна тривалість: {format_duration(total_popular_duration)}")

assert format_duration(187) == "3:07"
assert [t[0] for t in popular_tracks] == ["Ой у лузі червона калина", "Тримай", "Незалежність"]
assert total_popular_duration == 635
assert format_duration(total_popular_duration) == "10:35"
print("OK")

## ✅ Самоперевірка (5 запитань)

> Примітка: оригінальний шаблон уроку (`PY_UKR_L07_filled_template.docx`) — бінарний `.docx`-файл, який недоступний для автоматичного читання в цьому середовищі. Питання нижче побудовані на тому самому матеріалі конспекту (а не є дослівним перекладом файлу).

**1.** Функція не має `return`. Що буде у змінній, якщо зберегти результат виклику цієї функції?

<details><summary>Відповідь</summary><code>None</code> — Python автоматично повертає <code>None</code>, якщо в тілі функції немає явного <code>return</code>.</details>

**2.** `def greet(name, greeting="Привіт"):` — що надрукує `greet("Оля")` і чим це відрізняється від `greet("Оля", "Вітаю")`?

<details><summary>Відповідь</summary><code>greet("Оля")</code> використовує значення параметра за замовчуванням і виведе те саме, що й із явним другим аргументом <code>"Привіт"</code>. <code>greet("Оля", "Вітаю")</code> перевизначає <code>greeting</code> переданим значенням.</details>

**3.** Функція `add_item(lst, x): lst.append(x); return lst` — чиста чи нечиста? Чому?

<details><summary>Відповідь</summary>Нечиста — вона <b>мутує</b> вхідний список <code>lst</code> (побічний ефект поза функцією), а не повертає новий список.</details>

**4.** Навіщо розбивати `play_minesweeper_after` на п'ять окремих функцій замість одного суцільного блоку коду (як у `minesweeper_before_refactor.py`)?

<details><summary>Відповідь</summary>Принцип єдиної відповідальності: кожна функція відповідає за одну чітку задачу (згенерувати бомби, порахувати сусідів, надрукувати дошку тощо), її легше називати, тестувати окремо і повторно використовувати — а змінні кожної функції ізольовані у власному просторі імен, тому не конфліктують між собою (див. PREDICT вище).</details>

**5.** Задача: «з списку замовлень залишити тільки ті, що на суму більше 1000 грн». Це приклад predicate, transformer чи reducer?

<details><summary>Відповідь</summary>Predicate — для кожного елемента ставиться питання «так чи ні» (сума > 1000?), і за відповіддю елемент або залишається, або відкидається; кількість елементів на виході <b>менша або рівна</b> вхідній, форма самих елементів не змінюється.</details>

## Далі

Урок 8 — «Функції (продовження)» поглиблює цю тему: stack frame детальніше, `*args`, чисті функції як окрема практика, повний pipeline Filter → Map → Reduce. Конспект `lesson_07_functions/notes_functions.ipynb` можна переглянути вже зараз, якщо хочеться забігти наперед.